In [1]:
import json
from sentence_transformers import SentenceTransformer
import os
import numpy as np

/opt/anaconda3/envs/EnviOpti/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def extract_and_save_text_embeddings(descriptions_json_path='captions_MIR_DATASETS_B.json', output_dir='TEXT_EXTRACTED'):
    # 0. Vérifier la présence du fichier descriptions.json
    if not os.path.exists(descriptions_json_path):
        print(f"Erreur : Le fichier '{descriptions_json_path}' est introuvable. Veuillez vous assurer qu'il se trouve dans le même répertoire que ce script.")
        return

    try:
        with open(descriptions_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except json.JSONDecodeError:
        print(f"Erreur : Impossible de décoder le fichier JSON '{descriptions_json_path}'. Assurez-vous qu'il est correctement formaté.")
        return
    except Exception as e:
        print(f"Une erreur inattendue est survenue lors du chargement de '{descriptions_json_path}': {e}")
        return

    print(f"Nombre d'entrées chargées depuis '{descriptions_json_path}' : {len(data)}")
    if len(data) > 0:
        print(f"Exemple des 5 premières clés (noms d'images) : {list(data.keys())[0:5]}")
    else:
        print("Le fichier JSON est vide.")
        return

    # 2. Charger le modèle Sentence-BERT
    print("Chargement du modèle Sentence-BERT 'paraphrase-MiniLM-L6-v2'...")
    try:
        model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
        print("Modèle chargé avec succès.")
    except Exception as e:
        print(f"Erreur lors du chargement du modèle Sentence-BERT : {e}")
        print("Veuillez vérifier votre connexion internet ou installer le modèle manuellement si nécessaire.")
        return

    # 3. Créer le dossier de sortie pour les embeddings textuels
    os.makedirs(output_dir, exist_ok=True)
    print(f"Le dossier de sortie '{output_dir}' a été créé ou existe déjà.")

    processed_count = 0
    print(f"Extraction des embeddings textuels et sauvegarde dans '{output_dir}'...")

    for image_name, description in data.items():
        if not isinstance(description, str):
            print(f"Avertissement: La description pour '{image_name}' n'est pas une chaîne de caractères et sera ignorée.")
            continue

        try:
            embedding = model.encode(description)
            base_image_name = os.path.basename(image_name)
            safe_filename = os.path.splitext(base_image_name)[0].replace('.', '_').replace(os.sep, '_') + '.txt'
            output_filepath = os.path.join(output_dir, safe_filename)
            np.savetxt(output_filepath, embedding)
            processed_count += 1
        except Exception as e:
            print(f"Erreur lors du traitement de l'image '{image_name}' : {e}")

    print(f"----------------------------------------------------------------------")
    print(f"Processus terminé : {processed_count} embeddings textuels extraits et sauvegardés.")
    print(f"Vérifiez le dossier '{output_dir}' pour les fichiers de sortie.")

In [3]:
if __name__ == "__main__":
    extract_and_save_text_embeddings()

Nombre d'entrées chargées depuis 'captions_MIR_DATASETS_B.json' : 2665
Exemple des 5 premières clés (noms d'images) : ['oiseaux/vulture/2_0_oiseaux_vulture_1888.jpg', 'oiseaux/vulture/2_0_oiseaux_vulture_1863.jpg', 'oiseaux/vulture/2_0_oiseaux_vulture_1877.jpg', 'oiseaux/vulture/2_0_oiseaux_vulture_1876.jpg', 'oiseaux/vulture/2_0_oiseaux_vulture_1862.jpg']
Chargement du modèle Sentence-BERT 'paraphrase-MiniLM-L6-v2'...
Modèle chargé avec succès.
Le dossier de sortie 'TEXT_EXTRACTED' a été créé ou existe déjà.
Extraction des embeddings textuels et sauvegarde dans 'TEXT_EXTRACTED'...
----------------------------------------------------------------------
Processus terminé : 2665 embeddings textuels extraits et sauvegardés.
Vérifiez le dossier 'TEXT_EXTRACTED' pour les fichiers de sortie.
